In [1]:
import json
import numpy as np
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
import random
import pickle
from sklearn.metrics import precision_recall_fscore_support
from keras.models import load_model

ModuleNotFoundError: No module named 'tensorflow.keras'

In [ ]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	return ' '.join(filtered_tokens)
	
# Load JSON data
def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

[nltk_data] Downloading package punkt to /Users/chenluyao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')
evidence_map = load_data('data/curated/preprocessed_evidence_map.json')

NameError: name 'load_data' is not defined

In [5]:
data_for_dataframe = []
evidence_keys = list(evidence_map.keys())  # List of all evidence IDs

for claim_id, claim_details in train_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    claim_evidences = set(claim_details['evidences'])  # Convert to set for faster checks

    # Add positive examples
    for eid in claim_evidences:
        evidence_text = evidence_map.get(eid, "NULL")  
        if evidence_text != "NULL":
            data_for_dataframe.append({
                'claim': claim_text,
                'evidence': evidence_text,
                'label': 1  # Label as relevant
            })

    # Add negative examples
    num_neg_samples = min(len(claim_evidences), len(evidence_keys) - len(claim_evidences))  # Limit the number of negative samples
    negative_samples = random.sample([k for k in evidence_keys if k not in claim_evidences], num_neg_samples)
    for eid in negative_samples:
        evidence_text = evidence_map[eid]
        data_for_dataframe.append({
            'claim': claim_text,
            'evidence': evidence_text,
            'label': 0  # Label as not relevant
        })

train_df = pd.DataFrame(data_for_dataframe)

In [7]:
def text2seq(train_text, tokenizer_name):
	tokenizer = Tokenizer()
	tokenizer.fit_on_texts(train_text)
	input_text_index = tokenizer.word_index # return dictionary of wordss {'the':1, 'earth':2, 'is':3}

	with open(tokenizer_name+'.pickle', 'wb') as handle:
		pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

	max_length = max([len(s.split()) for s in train_text])
	print ("max length:", max_length)

	train_sequence = tokenizer.texts_to_sequences(train_text)
	return (train_sequence, input_text_index, max_length)

def to_padding(train_df):
	# Initialize and fit the tokenizer on claim and evidence separately
	x_claims_seq, x_claims_word_index, max_claims_length = text2seq(train_df["claim"].tolist(), "tokenizer_claims_full")
	x_sents_seq, x_sents_word_index, max_sents_length = text2seq(train_df["evidence"].tolist(), "tokenizer_evidence_full")

	x_claims_data = pad_sequences(x_claims_seq, maxlen=max_claims_length)  #returns array of data
	x_sents_data = pad_sequences(x_sents_seq, maxlen=max_sents_length)
	x_labels = train_df['label'].values

	return (x_claims_data, x_sents_data, x_labels, x_claims_word_index,  x_sents_word_index)

def create_embedding_matrix(vocab_size, word_vectors, word_index, embedding_dim):
	embedding_matrix = np.zeros((vocab_size, embedding_dim))
	for word, i in word_index.items():
		if word in word_vectors:
			embedding_vector = word_vectors[word]
			if embedding_vector is not None:
				embedding_matrix[i] = embedding_vector
	return embedding_matrix, embedding_dim

In [14]:
x_claim, x_sents, x_labels, x_claims_word_index,  x_sents_word_index = to_padding(train_df)
loss = 'binary_crossentropy'

print ("x claim word index ", len(x_claims_word_index))
print ("x sent word index ", len(x_sents_word_index))

vocab_size_claims = len(x_claims_word_index) + 1
vocab_size_evidences = len(x_sents_word_index) + 1

max length: 35
max length: 180
x claim word index  2748
x sent word index  14946


In [15]:
word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')

In [16]:
embedding_dim = 300  # dimension of word2vec vectors
(embed_matrix_claim, embed_dim_claim) = create_embedding_matrix(vocab_size_claims, word_vectors, x_claims_word_index, embedding_dim)
(embed_matrix_evidence, embed_dim_evidence) = create_embedding_matrix(vocab_size_evidences, word_vectors, x_sents_word_index, embedding_dim)

print ("embed_matrix_claim shape ", embed_matrix_claim.shape)
print ("embed_matrix_evidence shape ", embed_matrix_evidence.shape)

embed_matrix_claim shape  (2749, 300)
embed_matrix_evidence shape  (14947, 300)


In [17]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout, concatenate, BatchNormalization

# Define the model
def create_model(vocab_size_claims, vocab_size_evidences, maxlen_claims, maxlen_evidences, 
								embed_dim_claim, embed_dim_evidence):
		
	claims_input = Input(shape=(None,), dtype='int32', name='claims')
	embed_claims = Embedding(vocab_size_claims, embed_dim_claim)(claims_input)
	encoded_claims = LSTM(256, return_sequences=True)(embed_claims)
	encoded_claims = LSTM(16)(encoded_claims)
	encoded_claims = BatchNormalization()(encoded_claims)
		
	evidences_input = Input(shape=(None,), dtype='int32', name='evidences')
	embed_evidences = Embedding(vocab_size_evidences, embed_dim_evidence)(evidences_input)
	encoded_evidences = LSTM(256, return_sequences=True)(embed_evidences)
	encoded_evidences= LSTM(64)(encoded_evidences)
	encoded_evidences = BatchNormalization()(encoded_evidences)
		
	concatenate_layers = concatenate([encoded_claims, encoded_evidences], axis=-1)
	concatenate_layers = Dropout(0.5)(concatenate_layers)
	concatenate_layers = Dense(64, activation='relu')(concatenate_layers)
	pred_label = Dense(1, activation='sigmoid')(concatenate_layers)

	model = Model(inputs=[claims_input, evidences_input], outputs=pred_label)
	model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

	return model


max_claims_length = 35
max_sents_length = 180

model = create_model(vocab_size_claims, vocab_size_evidences, max_claims_length, max_sents_length, 
								embed_dim_claim, embed_dim_evidence)

model.layers[2].set_weights([embed_matrix_claim])
model.layers[2].trainable = False
model.layers[3].set_weights([embed_matrix_evidence])
model.layers[3].trainable = False

print(model.summary())


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ claims (InputLayer) │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ evidences           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, None, 300) │    824,700 │ claims[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, None, 300) │  4,484,100 │ evidences[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_8 (LSTM)       │ (None, None, 256) │    570,368 │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ (None, None, 256) │    570,368 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ (None, 16)        │     17,472 │ lstm_8[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ (None, 64)        │     82,176 │ lstm_10[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16)        │         64 │ lstm_9[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ lstm_11[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 80)        │          0 │ batch_normalizat… │
│ (Concatenate)       │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 80)        │          0 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      5,184 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         65 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,554,753 (25.00 MB)

 Trainable params: 1,245,793 (4.75 MB)

 Non-trainable params: 5,308,960 (20.25 MB)

None


In [20]:
from keras.callbacks import EarlyStopping, ModelCheckpoint, Callback, CSVLogger
early_stopping = EarlyStopping(monitor='val_loss', patience=2)

csv_logger = CSVLogger('lstm_training_tf.log')
model_path = 'full_lstm_evidence_retrieval.hdf5'
history = model.fit({'claims': x_claim, 'evidences': x_sents}, x_labels, 
						epochs=60, batch_size=64, validation_split=0.12, callbacks=[early_stopping, csv_logger,
						ModelCheckpoint(filepath=model_path, monitor='val_loss', save_best_only=True)])	

ValueError: The filepath provided must end in `.keras` (Keras model format). Received: filepath=full_lstm_evidence_retrieval.hdf5

In [ ]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'evidence': eids
        })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df 